In [1]:
# extract biodiesel plant dataset
from ca_biositing.pipeline.etl.extract.biodiesel_plants import extract as extract_biodiesel

biodiesel_df = extract_biodiesel("../../../../../../resources/prefect/")

19:38:18.866 | INFO    | Task run 'extract' - Extracting raw data from 'Biodiesel_Plants.csv'...

19:38:20.519 | INFO    | Task run 'extract' - Successfully extracted raw data.

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


19:38:20.536 | INFO    | Task run 'extract' - Finished in state Completed()

In [2]:
# extract geocoded biodiesel data
import gspread
from ca_biositing.pipeline.etl.extract.factory import create_extractor

GSHEET_NAME = "address-to-geocoded"
WORKSHEET_NAME = "Biodiesel Plants"

extract = create_extractor(GSHEET_NAME, WORKSHEET_NAME)


In [3]:
geocoded_df = extract("../../../../../../resources/prefect/")

19:38:23.009 | INFO    | Task run 'extract_biodiesel plants' - Extracting raw data from 'Biodiesel Plants' in 'address-to-geocoded'...

DEBUG: gsheet_to_df called for address-to-geocoded / Biodiesel Plants
DEBUG: Authenticating with ../../../../../../resources/prefect/credentials.json
DEBUG: Opening spreadsheet address-to-geocoded
DEBUG: Opening worksheet by name: Biodiesel Plants
DEBUG: Fetching all values from worksheet
DEBUG: Successfully fetched 79 rows


19:38:25.161 | INFO    | Task run 'extract_biodiesel plants' - Successfully extracted raw data from Biodiesel Plants.

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


19:38:25.178 | INFO    | Task run 'extract_biodiesel plants' - Finished in state Completed()

In [4]:
import numpy as np
# replace whitespace with NaN
geocoded_df = geocoded_df.replace(r'^\s*$', np.nan, regex=True)
geocoded_df

C:\Users\Abigail\AppData\Local\Temp\ipykernel_17580\240408571.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  geocoded_df = geocoded_df.replace(r'^\s*$', np.nan, regex=True)


,company,city,state,address,latitude,longitude,geocoded_status,closest_address_line_1,closest_address_line_2,closest_city,...,closest_latitude,closest_longitude,closest_geoid,closest_state_name,closest_state_fips,closest_county_name,closest_county_fips,is_na,address_id,merged_address
0,american greenfuels,new haven,connecticut,NaN,41.2901,-72.9029,false,30 Waterfront Street,NaN,New Haven,...,41.289613,-72.9039114,00000,CT,00,South Central Connecticut Planning Region,000,FALSE,0,"american greenfuels, new haven, connecticut,"
1,down to earth energy llc,monroe,georgia,NaN,33.75717,-83.7277,true,941 Monroe Jersey Road Southeast,NaN,Monroe,...,33.7574384,-83.7282358,13297,GA,13,Walton,297,FALSE,589,"down to earth energy llc, monroe, georgia,"
2,maine bio-fuel inc,portland,maine,NaN,43.6914,-70.3281,true,51 Ingersoll Drive,NaN,Portland,...,43.6919484,-70.3281613,23005,ME,23,Cumberland,005,FALSE,590,"maine bio-fuel inc, portland, maine,"
3,cape cod biofuels inc,sandwich,massachusetts,NaN,41.7177,-70.4845,true,14 Jan Sebastian Drive,NaN,Sandwich,...,41.7177097,-70.4845026,25001,MA,25,Barnstable,001,FALSE,497,"cape cod biofuels inc, sandwich, massachusetts,"
4,renewable fuels by peterson,north haverhill,new hampshire,NaN,44.077,-72.0047,true,NaN,NaN,Haverhill,...,44.0839848,-72.0201877,33009,NH,33,Grafton,009,FALSE,591,"renewable fuels by peterson, north haverhill, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,walsh biofuels llc,mauston,wi,"wi-80 & mauston rd, mauston, wi 53948",43.77770321,-90.0551531,true,NaN,NaN,Mauston,...,43.7777045,-90.0551353,55057,WI,55,Juneau,057,FALSE,136,"walsh biofuels llc, mauston, wi, wi-80 & maust..."
74,western iowa energy - agron bioenergy,watsonville,ca,"860 w beach st, watsonville, ca 95076",36.90388067,-121.7696898,true,860 West Beach Street,NaN,Watsonville,...,36.9038807,-121.7696791,06087,CA,06,Santa Cruz,087,FALSE,368,"western iowa energy - agron bioenergy, watsonv..."
75,white mountain biodiesel llc,north haverhill,nh,"35 business pk rd, north haverhill, nh 03774",44.07671166,-72.0049136,true,35 Business Park Road,NaN,Haverhill,...,44.0767194,-72.0051389,33009,NH,33,Grafton,009,FALSE,591,"white mountain biodiesel llc, north haverhill,..."
76,world energy - natchez,natchez,ms,"l e barry rd, natchez, ms 39120",31.53322759,-91.43745856,true,L E Barry Road,NaN,Natchez,...,31.5332459,-91.4374371,28001,MS,28,Adams,001,FALSE,628,"world energy - natchez, natchez, ms, l e barry..."


In [5]:
#Modified ETL Transform for Biodiesel.

import pandas as pd
import numpy as np
from typing import List, Optional, Dict
from prefect import task, get_run_logger
from ca_biositing.pipeline.utils.cleaning_functions import cleaning as cleaning_mod
from ca_biositing.pipeline.utils.cleaning_functions import coercion as coercion_mod
from ca_biositing.pipeline.utils.name_id_swap import normalize_dataframes
from ca_biositing.pipeline.utils.geo_utils import parse_addresses


# --- CONFIGURATION ---
# List the names of the extract modules this transform depends on.
# The pipeline runner provides these in the `data_sources` dictionary.
EXTRACT_SOURCES: List[str] = ["biodiesel_plants"]

# List the unique address information needed to find the geocoded address.
MERGE_COLUMNS = ["company", "city", "state", "address"]

# don't edit
geocoded_columns = ["geocoded_status", "closest_address_line_1", "closest_address_line_2", "closest_city", "closest_county", "closest_state", "closest_postal_code", "closest_latitude", "closest_longitude", "closest_geoid", "closest_state_name", "closest_state_fips", "closest_county_name", "closest_county_fips","address_id"]

@task
def transform(
    data_sources: Dict[str, pd.DataFrame],
    etl_run_id: int = None,
    lineage_group_id: int = None
) -> Optional[pd.DataFrame]:
    """
    Transforms raw data from multiple sources into a structured format.

    Args:
        data_sources: Dictionary where keys are source names and values are DataFrames.
        etl_run_id: ID of the current ETL run.
        lineage_group_id: ID of the lineage group.
    """
    try:
        logger = get_run_logger()
    except Exception:
        import logging
        logger = logging.getLogger(__name__)

    # CRITICAL: Lazy import models inside the task to avoid Docker import hangs
    from ca_biositing.datamodels.models import (
        LocationAddress,
        Place
        # Add other models needed for normalization here (e.g., Resource, Unit)
    )

    # 1. Input Validation
    for source_name in EXTRACT_SOURCES:
        if source_name not in data_sources:
            logger.error(f"Required data source '{source_name}' not found.")
            return None

    logger.info(f"Transforming data from sources: {EXTRACT_SOURCES}")

    # 2. Cleaning & Coercion
    processed_dfs = []
    for source_name in EXTRACT_SOURCES:
        df = data_sources[source_name].copy()

        if df.empty:
            continue

        # Standardize column names (snake_case) and basic string cleaning
        cleaned_df = cleaning_mod.standard_clean(df)

        # Add lineage tracking metadata
        cleaned_df['etl_run_id'] = etl_run_id
        cleaned_df['lineage_group_id'] = lineage_group_id

        # if address = null add state + city
        # cleaned_df["address"] = np.where(cleaned_df["address"].isna(), cleaned_df["city"] + " " + cleaned_df["state"], cleaned_df["address"])

        # Coerce data types (Update these lists based on your schema)
        coerced_df = coercion_mod.coerce_columns(
            cleaned_df,
            int_cols=["capacity_mmg_per_y", "bbi_index"],
            float_cols=[],
            datetime_cols=['created_at', 'updated_at']
        )

        processed_dfs.append(coerced_df)

    if not processed_dfs:
        return pd.DataFrame()

    # Combine sources if necessary, or handle them individually
    combined_df = pd.concat(processed_dfs, ignore_index=True)

    # added_address_df = pd.concat([combined_df, address_df, geoid_df], axis=1)

    GEOCODED_DF_FILTER = MERGE_COLUMNS + geocoded_columns

    added_address_df = pd.merge(combined_df, geocoded_df[GEOCODED_DF_FILTER], on=MERGE_COLUMNS, how='left')

    # 3. Normalization (Name-to-ID Swapping)
    # Format: 'dataframe_column': (SQLAlchemyModel, 'lookup_field_in_db')
    normalize_columns = {

    }


    # Manual normalization for Place (County) to avoid NotNullViolation on geoid
    # and provide a resilient lookup that defaults to state-level GEOID.
    from ca_biositing.pipeline.utils.geo_utils import get_geoid
    from sqlmodel import Session, select
    from ca_biositing.pipeline.utils.engine import engine

    with Session(engine) as session:
        places = session.exec(select(Place.geoid, Place.county_name)).all()
        county_to_geoid = {p.county_name.lower(): p.geoid for p in places if p.county_name}

    logger.info("Normalizing data (swapping names for IDs)...")
    normalized_df = normalize_dataframes(added_address_df, normalize_columns)[0]


    # Bridge County (Place) to LocationAddress
    # We need to find or create a generic LocationAddress for each County
    if 'closest_geoid' in normalized_df.columns:
        logger.info("Bridging County (Place) to LocationAddress...")
        from sqlmodel import Session, select
        from ca_biositing.pipeline.utils.engine import engine

        with Session(engine) as session:
            # Get unique county_ids (these are geoids from Place table)
            county_ids = normalized_df['closest_geoid'].dropna().unique()
            place_to_address_map = {}

            for index, row in normalized_df.iterrows():
                # Find or create LocationAddress and Place where geography_id = geoid
                geoid = row["closest_geoid"]
                if geoid is not None and geoid != "":
                    stmt1 = select(Place).where(
                        Place.geoid == geoid
                    )
                    place = session.exec(stmt1).first()

                    stmt2 = select(LocationAddress).where(
                        LocationAddress.geography_id == geoid,
                        # LocationAddress.address_line1 == None
                    )
                    address = session.exec(stmt2).first()

                    if not place:
                        logger.info(f"Creating new Place for county geoid: {geoid}")
                        place = Place(
                            geoid=geoid,
                            state_name=row["closest_state_name"],
                            state_fips=row["closest_state_fips"],
                            county_name=row["closest_county_name"],
                            county_fips=row["closest_county_fips"],
                        )
                        session.add(place)
                        session.flush()

                    if not address:
                        logger.info(f"Creating new generic LocationAddress for county geoid: {geoid}")

                        address = LocationAddress(
                            geography_id=geoid,
                            address_line1=row["closest_address_line_1"],
                            address_line2=row["closest_address_line_2"],
                            city=row["closest_city"],
                            zip=row["closest_postal_code"],
                            lat=row["closest_latitude"],
                            lon=row["closest_longitude"],
                            is_anonymous=False
                            )
                        session.add(address)
                        session.flush()

                    place_to_address_map[geoid] = address.id

            session.commit()

            # Map county_id (Place.geoid) to sampling_location_id (LocationAddress.id)
            normalized_df['address_id'] = normalized_df['closest_geoid'].map(place_to_address_map)
            logger.info(f"Mapped {len(place_to_address_map)} counties to LocationAddresses")

    print(normalized_df)


    # 5. Final Mapping & Selection
    # TODO: Update this list to match the columns in your target database table
    try:
        # Ensure lineage columns exist even if not provided in input
        if etl_run_id:
            normalized_df['etl_run_id'] = etl_run_id
        if lineage_group_id:
            normalized_df['lineage_group_id'] = lineage_group_id

        print(normalized_df.columns)

        final_df = normalized_df[[
            "company",
            "bbi_index",
            "city",
            "state",
            "capacity_mmg_per_y",
            "feedstock",
            "status",
            "address_id",
            "coordinates",
            "latitude",
            "longitude",
            "source",
            'etl_run_id',
            'lineage_group_id',
        ]].copy()

        logger.info(f"Successfully transformed {len(final_df)} records.")
        return final_df

    except KeyError as e:
        logger.error(f"Missing required column during transform: {e}")
        return normalized_df

transformed_df = transform(data_sources={"biodiesel_plants": biodiesel_df},)
transformed_df

19:38:39.229 | INFO    | Task run 'transform' - Transforming data from sources: ['biodiesel_plants']

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\src\ca_biositing\pipeline\ca_biositing\pipeline\utils\cleaning_functions\cleaning.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.astype("object").replace(regex, np.nan, regex=True)


19:38:42.613 | INFO    | Task run 'transform' - Normalizing data (swapping names for IDs)...

19:38:42.623 | INFO    | Task run 'transform' - Bridging County (Place) to LocationAddress...

19:38:42.676 | INFO    | Task run 'transform' - Creating new Place for county geoid: 00000

...

                                  company  bbi_index             city  \
0                     american greenfuels       <NA>        new haven   
1                down to earth energy llc       <NA>           monroe   
2                      maine bio-fuel inc       <NA>         portland   
3                   cape cod biofuels inc       <NA>         sandwich   
4             renewable fuels by peterson       <NA>  north haverhill   
..                                    ...        ...              ...   
73                     walsh biofuels llc         58          mauston   
74  western iowa energy - agron bioenergy         60      watsonville   
75           white mountain biodiesel llc         62  north haverhill   
76                 world energy - natchez         65          natchez   
77                    world energy - rome         66             rome   

            state  capacity_mmg_per_y            feedstock       status  \
0     connecticut                  35           

19:38:45.182 | INFO    | Task run 'transform' - Successfully transformed 78 records.

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


19:38:45.197 | INFO    | Task run 'transform' - Finished in state Completed()

,company,bbi_index,city,state,capacity_mmg_per_y,feedstock,status,address_id,coordinates,latitude,longitude,source,etl_run_id,lineage_group_id
0,american greenfuels,<NA>,new haven,connecticut,35,<NA>,<NA>,1,<NA>,41.2901,-72.9029,https://atlas.eia.gov/datasets/79dad60ce89c475...,None,None
1,down to earth energy llc,<NA>,monroe,georgia,2,<NA>,<NA>,2,<NA>,33.75717,-83.7277,https://atlas.eia.gov/datasets/79dad60ce89c475...,None,None
2,maine bio-fuel inc,<NA>,portland,maine,1,<NA>,<NA>,3,<NA>,43.6914,-70.3281,https://atlas.eia.gov/datasets/79dad60ce89c475...,None,None
3,cape cod biofuels inc,<NA>,sandwich,massachusetts,1,<NA>,<NA>,4,<NA>,41.7177,-70.4845,https://atlas.eia.gov/datasets/79dad60ce89c475...,None,None
4,renewable fuels by peterson,<NA>,north haverhill,new hampshire,8,<NA>,<NA>,5,<NA>,44.077,-72.0047,https://atlas.eia.gov/datasets/79dad60ce89c475...,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,walsh biofuels llc,58,mauston,wi,5,distillers corn oil,operational,65,"43.777703209506015, -90.05515309603231",43.777703,-90.055153,https://issuu.com/bbiinternational/docs/biodie...,None,None
74,western iowa energy - agron bioenergy,60,watsonville,ca,15,multifeedstock,operational,66,"36.90388067222262, -121.76968983169887",36.903881,-121.76969,https://issuu.com/bbiinternational/docs/biodie...,None,None
75,white mountain biodiesel llc,62,north haverhill,nh,8,yellow grease,operational,5,"44.07671166339629, -72.0049135970569",44.076712,-72.004914,https://issuu.com/bbiinternational/docs/biodie...,None,None
76,world energy - natchez,65,natchez,ms,72,vegetable oils,operational,67,"31.533227585047626, -91.4374585606938",31.533228,-91.437459,https://issuu.com/bbiinternational/docs/biodie...,None,None
